### Running PageIndex Locally

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.home() / "Projects" / "RAG_v1" / ".env")

os.environ["NVIDIA_NIM_API_KEY"] = os.environ["NVIDIA_API_KEY"]

print("cwd:", os.getcwd())
print("NVIDIA key loaded:", bool(os.environ["NVIDIA_NIM_API_KEY"]))
# Then re-run that cell (sets the var in the live kernel) and re-run the submit loop. You should see NVIDIA key loaded: True.
# Quick sanity check that the whole local path is healthy:

import litellm

print(litellm.validate_environment("nvidia_nim/meta/llama-3.1-8b-instruct"))
# → {'keys_in_environment': True, 'missing_keys': []}

cwd: /home/seeker/Projects/RAG_v1
NVIDIA key loaded: True
{'keys_in_environment': True, 'missing_keys': []}


In [3]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.home() / "Projects" / "RAG_v1" / ".env")

os.environ["NVIDIA_NIM_API_KEY"] = os.environ["NVIDIA_API_KEY"]

import pageindex.utils as _u

# 1) Don't hammer NVIDIA's free tier → no 429s
_u.SUMMARY_CONCURRENCY = 2

# 2) Cap oversized summary prompts (PORT_PACKAGE has a 200k-char node) → no 504s
_orig_acomp = _u.llm_acompletion


async def _capped(model, prompt):
    if len(prompt) > 6000:
        prompt = prompt[:6000] + "\n...[truncated]"
    return await _orig_acomp(model, prompt)


_u.llm_acompletion = _capped

from pageindex import PageIndexLocalClient, utils

pi_client = PageIndexLocalClient(
    model="nvidia_nim/meta/llama-3.1-8b-instruct",
    summary_model="nvidia_nim/meta/llama-3.1-8b-instruct",
    retrieve_model="nvidia_nim/meta/llama-3.1-8b-instruct",
    storage_path=".pageindex",
)
print(pi_client.__class__.__name__)


PageIndexLocalClient


### Building The JSON Tree

In [ ]:
PDF_SOURCE_DIR = "Policy Documents Curated 15"
REGISTRY_PATH = "pdf_registry.json"

pdf_files = sorted(
    os.path.join(PDF_SOURCE_DIR, f)
    for f in os.listdir(PDF_SOURCE_DIR)
    if f.lower().endswith(".pdf")
)
print(f"Found {len(pdf_files)} PDFs")

existing = {d["name"]: d["id"] for d in pi_client.list_documents()["documents"]}
doc_ids = {}
for pdf_path in pdf_files:
    filename = os.path.basename(pdf_path)
    if filename in existing:
        doc_ids[filename] = existing[filename]
        print(f"⏭️  Already indexed: {filename}")
        continue
    doc_ids[filename] = pi_client.submit_document(pdf_path)["doc_id"]  # synchronous
    print(f"✅ Submitted: {filename} → {doc_ids[filename]}")

print("\nAll doc_ids:", doc_ids)

Found 15 PDFs
⏭️  Already indexed: 14..Trade credit insc_GEN756.pdf
⏭️  Already indexed: Arogya_Sanjeevani_Policy_Wording_KEN_034037d936.pdf
⏭️  Already indexed: Auto_Secure_Commercial_Vehicle_Package_Policy_Base_Policy_Wording_22b7905015.pdf
⏭️  Already indexed: Bharat_Griha_Raksha_Policy_Policy_Wordings_5219f40e18.pdf
⏭️  Already indexed: Click-2-Protect-Optima-Secure-Policy-Bond-101Y122V05.pdf
⏭️  Already indexed: Cyber_Shield_Policy_Wordings_78baa23b5a.pdf
⏭️  Already indexed: PORT_PACKAGE_Policy_wording_a5f317091b.pdf
⏭️  Already indexed: Policy_Wordings_aviation_insurance.pdf_7b7e60200a.pdf
⏭️  Already indexed: Policy_Wordings_contractors_plant_and_machinery_insurance.pdf_0175bcd067.pdf
⏭️  Already indexed: Policy_Wordings_political_risk_insurance_for_investors.pdf_b7a0e7c805.pdf
⏭️  Already indexed: SBI_General_Livestock_Policy_Wording_38ef0b201b.pdf
⏭️  Already indexed: Weather_Insurance_Policy_Wordings_Retail_6a1bd806a7.pdf
⏭️  Already indexed: hdfc-life-smart-pension-plus-v13

In [11]:
import json

print(json.dumps(doc_ids, indent=4))

{
    "14..Trade credit insc_GEN756.pdf": "pi-58068b94fd464bedaf251e3b442d3183",
    "Arogya_Sanjeevani_Policy_Wording_KEN_034037d936.pdf": "pi-7a77210c58344134988c513ce697a93a",
    "Auto_Secure_Commercial_Vehicle_Package_Policy_Base_Policy_Wording_22b7905015.pdf": "pi-66af3e82285e4e24819dcc87fcda1343",
    "Bharat_Griha_Raksha_Policy_Policy_Wordings_5219f40e18.pdf": "pi-a47f79df4ac040f5984d512c4e1225ae",
    "Click-2-Protect-Optima-Secure-Policy-Bond-101Y122V05.pdf": "pi-12de45f887e644009982f039d294eb7e",
    "Cyber_Shield_Policy_Wordings_78baa23b5a.pdf": "pi-6392aba4cb2f45ce8abf2f75871f3e93",
    "PORT_PACKAGE_Policy_wording_a5f317091b.pdf": "pi-da50dbffde964da4ba36ede6c7079e11",
    "Policy_Wordings_aviation_insurance.pdf_7b7e60200a.pdf": "pi-155f96f798b24b41982378c2bef596a5",
    "Policy_Wordings_contractors_plant_and_machinery_insurance.pdf_0175bcd067.pdf": "pi-9d8ead02fe1f4f6284671bf58c07e80b",
    "Policy_Wordings_political_risk_insurance_for_investors.pdf_b7a0e7c805.pdf": "pi-